In [9]:
# %% [markdown]
# # Task 1: News Topic Classifier using DistilBERT (Pure PyTorch Dataset - No Errors)

# %% [code]
# 1. Install required libraries
!pip install -q transformers scikit-learn accelerate gradio pandas pyarrow torch

# %% [code]
# 2. Imports
import numpy as np
import pandas as pd
import requests
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import (
    DistilBertTokenizerFast,
    DistilBertForSequenceClassification,
    Trainer,
    TrainingArguments
)
from sklearn.metrics import accuracy_score, f1_score

# %% [code]
# 3. Download AG News dataset as Parquet files (same as before)
print("Downloading AG News dataset...")

train_url = "https://huggingface.co/datasets/ag_news/resolve/main/data/train-00000-of-00001.parquet"
test_url = "https://huggingface.co/datasets/ag_news/resolve/main/data/test-00000-of-00001.parquet"

def download_parquet(url):
    response = requests.get(url)
    response.raise_for_status()
    with open("temp.parquet", "wb") as f:
        f.write(response.content)
    return pd.read_parquet("temp.parquet")

train_df = download_parquet(train_url)
test_df = download_parquet(test_url)

train_df = train_df.head(5000)
test_df = test_df.head(1000)

print(f"Train size: {len(train_df)}, Test size: {len(test_df)}")
print(f"Sample text: {train_df['text'].iloc[0][:100]}...")
print(f"Sample label: {train_df['label'].iloc[0]}")

# %% [code]
# 4. Tokenizer
model_name = "distilbert-base-uncased"
tokenizer = DistilBertTokenizerFast.from_pretrained(model_name)

# Tokenize all texts once (outside training loop)
def tokenize_texts(texts):
    return tokenizer(texts, padding="max_length", truncation=True, max_length=128, return_tensors=None)

# Apply tokenization to DataFrames
train_encodings = tokenize_texts(train_df['text'].tolist())
test_encodings = tokenize_texts(test_df['text'].tolist())

# Extract labels
train_labels = train_df['label'].tolist()
test_labels = test_df['label'].tolist()

# %% [code]
# 5. Custom PyTorch Dataset (no datasets library)
class NewsDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

    def __len__(self):
        return len(self.labels)

train_dataset = NewsDataset(train_encodings, train_labels)
test_dataset = NewsDataset(test_encodings, test_labels)

# %% [code]
# 6. Load model
model = DistilBertForSequenceClassification.from_pretrained(model_name, num_labels=4)

# %% [code]
# 7. Metrics function
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    preds = np.argmax(predictions, axis=1)
    acc = accuracy_score(labels, preds)
    f1 = f1_score(labels, preds, average="weighted")
    return {"accuracy": acc, "f1_weighted": f1}

# %% [code]
# 8. Training arguments
training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=1,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=10,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
)

# %% [code]
# 9. Trainer (no tokenizer argument, works with custom Dataset)
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics,
)

# %% [code]
# 10. Train
print("Starting training...")
trainer.train()

# %% [code]
# 11. Evaluate on test set
eval_results = trainer.evaluate()
print(f"\nTest Accuracy: {eval_results['eval_accuracy']:.4f}")
print(f"Test F1 (weighted): {eval_results['eval_f1_weighted']:.4f}")

# %% [code]
# 12. Save model and tokenizer
model.save_pretrained("./saved_model")
tokenizer.save_pretrained("./saved_model")
print("Model saved to ./saved_model")

# Determine the device the model is on
device = model.device  # will be 'cuda:0' if GPU available

def predict(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=128)
    # Move inputs to the same device as the model
    inputs = {k: v.to(device) for k, v in inputs.items()}
    outputs = model(**inputs)
    pred = outputs.logits.argmax(dim=1).item()
    labels_map = {0: "World", 1: "Sports", 2: "Business", 3: "Sci/Tech"}
    return labels_map[pred]

test_headlines = [
    "Apple releases new iPhone with advanced camera",
    "Manchester United wins Premier League title",
    "Stock market hits all-time high amid tech rally",
    "Scientists discover new species in Amazon rainforest"
]

print("\nSample predictions:")
for headline in test_headlines:
    print(f"Headline: {headline[:50]}... -> {predict(headline)}")

Train size: 5000, Test size: 1000
Sample text: Wall St. Bears Claw Back Into the Black (Reuters) Reuters - Short-sellers, Wall Street's dwindling\b...
Sample label: 2


[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Starting training...


Epoch,Training Loss,Validation Loss,Accuracy,F1 Weighted
1,0.295475,0.328615,0.895000,0.893942


Training Loss,Validation Loss,Epoch,Accuracy,F1 Weighted
0.295475,0.328615,1,0.895000,0.893942



Test Accuracy: 0.8950
Test F1 (weighted): 0.8939
Model saved to ./saved_model

Sample predictions:
Headline: Apple releases new iPhone with advanced camera... -> Sci/Tech
Headline: Manchester United wins Premier League title... -> Sports
Headline: Stock market hits all-time high amid tech rally... -> Business
Headline: Scientists discover new species in Amazon rainfore... -> Sci/Tech


In [10]:
import os
import shutil
from google.colab import files

# Check if the saved_model folder exists
if os.path.exists('./saved_model'):
    print("✅ Model found at './saved_model'")
    # List contents
    print("Contents:", os.listdir('./saved_model'))

    # Zip and download
    shutil.make_archive('saved_model', 'zip', './saved_model')
    files.download('saved_model.zip')
else:
    print("❌ Model not found. Did the training complete successfully?")

✅ Model found at './saved_model'
Contents: ['tokenizer.json', 'tokenizer_config.json', 'model.safetensors', 'config.json']


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>